In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [3]:
code = 'URE'
market = 'TO'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [4]:
dsv_timeseries_df = request_api.get_stock_time_series_data(
    code=code,
    market=market,
    start=start,
    end=end
)
dsv_timeseries_df

取得件数: 5097


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1167008,URE,TO,2006-01-19,1.34,1.40,1.32,1.38,369800,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1167009,URE,TO,2006-01-20,1.42,1.44,1.38,1.40,286100,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1167010,URE,TO,2006-01-23,1.35,1.41,1.35,1.41,287300,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1167011,URE,TO,2006-01-24,1.31,1.36,1.26,1.35,434100,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1167012,URE,TO,2006-01-25,1.31,1.38,1.27,1.35,393300,1.378,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5092,1172100,URE,TO,2026-05-04,2.39,2.51,2.39,2.45,313800,2.386,...,2.371954,2.067246,True,NaN,NaN,NaN,NaN,NaN,NaN,False
5093,1172101,URE,TO,2026-05-05,2.37,2.46,2.34,2.43,289100,2.396,...,2.386499,2.090301,True,NaN,NaN,NaN,NaN,NaN,NaN,False
5094,1172102,URE,TO,2026-05-06,2.53,2.53,2.34,2.40,406900,2.406,...,2.395422,2.117378,True,NaN,NaN,NaN,NaN,NaN,NaN,False
5095,1172103,URE,TO,2026-05-07,2.47,2.57,2.47,2.52,331600,2.446,...,2.417486,2.129714,True,NaN,NaN,NaN,NaN,NaN,NaN,False


In [5]:
def stock_prices_and_material_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_mat1: pd.DataFrame | None = None,
        df_mat2: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # mat1価格を統合
    if df_mat1 is not None:
        df_mat1_tmp = df_mat1.copy() if df_mat1 is not None else pd.DataFrame()
        if "date" not in df_mat1_tmp.columns:
            df_mat1_tmp = df_mat1_tmp.reset_index()
        df_mat1_tmp["date"] = pd.to_datetime(df_mat1_tmp["date"])
        df_mat1_tmp = df_mat1_tmp.set_index("date")
        df_mat1_tmp = df_mat1_tmp.loc[start:end]

    # mat2価格を統合
    if df_mat2 is not None:
        df_mat2_tmp = df_mat2.copy() if df_mat2 is not None else pd.DataFrame()
        if "date" not in df_mat2_tmp.columns:
            df_mat2_tmp = df_mat2_tmp.reset_index()
        df_mat2_tmp["date"] = pd.to_datetime(df_mat2_tmp["date"])
        df_mat2_tmp = df_mat2_tmp.set_index("date")
        df_mat2_tmp = df_mat2_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_mat1 is not None:
        df["MA5_MAT1"] = df_mat1_tmp["ma5"].reindex(df.index)
        df["MA25_MAT1"] = df_mat1_tmp["ma25"].reindex(df.index)
    if df_mat2 is not None:
        df["MA5_MAT2"] = df_mat2_tmp["ma5"].reindex(df.index)
        df["MA25_MAT2"] = df_mat2_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT1（右軸） ---
    if df_mat1 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT1"],
                name="MAT1_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT1"],
                name="MAT1_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT2（左軸） ---
    if df_mat2 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT2"],
                name="MAT2_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT2"],
                name="MAT2_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [6]:
name = "Silver Mountain Resources Inc"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_material_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_mat1=None,
    df_mat2=None
)
fig.show()

取得件数: 531


In [7]:
response = request_api.update_corp_finance_data(
    code=code,
    market=market
)
response

{'result': True}

In [9]:
ure_financials_data = request_api.get_corp_financials_data(code=code, market=market)
ure_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=code, market=market)
ure_cash_flow_data = request_api.get_corp_cash_flow_data(code=code, market=market)
ure_earnings_data = request_api.get_corp_earnings_data(code=code, market=market)
ure_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=code, market=market)

In [10]:
# ４年分の財務データ
ure_financials_data_df = pd.DataFrame(ure_financials_data['results'])
# ４年分のバランスシート
ure_balance_sheet_data_df = pd.DataFrame(ure_balance_sheet_data['results'])
# ４年分のキャッシュフロー
ure_cash_flow_data_df = pd.DataFrame(ure_cash_flow_data['results'])
# ４年分の収益データ
ure_earnings_data_df = pd.DataFrame(ure_earnings_data['results'])
# ４年分の四半期収益データ
ure_quarterly_earnings_data_df = pd.DataFrame(ure_quarterly_earnings_data['results'])

In [11]:
"""
◆ 1. 株価・市場データ
• 現在株価（Price）
• 時価総額（Market Cap）
• 出来高（Volume）
• 52週高値・安値
• Beta（ボラティリティ指標）ß
"""
stock_prices_market_data.stock_prices_and_market_data(
    code=code,
    market=market,
    bs_df=ure_balance_sheet_data_df
)

取得件数: 458
取得件数: 461
取得件数: 457
取得件数: 460
取得件数: 457
取得件数: 458
取得件数: 458
取得件数: 459
取得件数: 458
取得件数: 458


,close,market_cap,shares_outstanding,higher_rate_par_52_weeks,lower_rate_par_52_weeks,beta
0,0.67,NaN,NaN,2.72,0.39,1.163326
1,1.38,3.100855e+08,224699621.0,2.72,1.23,1.757082
2,2.25,6.095225e+08,270898900.0,2.49,1.13,1.449835
3,1.47,5.352285e+08,364101038.0,2.72,1.13,1.025857
4,2.06,7.790296e+08,378169709.0,3.30,0.78,1.554173


In [12]:
"""
◆ 2. 財務データ（Financials）+ EPS（Earnings Per Share）+ PBR（Price-to-Book Ratio）
• 売上高（Revenue）
• 営業利益（Operating Income）
• 純利益（Net Income）
• EBITDA（企業による）
• 総資産（Total Assets）
• 総負債（Total Liabilities）
• 現金（Cash）
• 希釈EPS（Diluted EPS）
• 基本EPS（Basic EPS）
• 営業キャッシュフロー（Operating Cash Flow）
• フリーキャッシュフロー（Free Cash Flow）
"""
financial_df = financial.calc_financial(
    code = code,
    market = market,
)
financial_df

取得件数: 2042


,date,revenue,earnings,total_assets,total_debt,cash_and_cash_equivalents,EBITDA,operating_income,basic_eps,diluted_eps,operating_cash_flow,free_cash_flow
0,2021-12-31,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1,2022-12-31,19000.0,-17140000.0,107895000.0,11076000.0,33003000.0,-13662000.0,-19794000.0,-0.08,-0.08,-18091000.0,-18800000.0
2,2023-12-31,17679000.0,-30656000.0,128376000.0,6543000.0,59700000.0,-27861000.0,-30842000.0,-0.12,-0.12,-16982000.0,-19021000.0
3,2024-12-31,33706000.0,-53189000.0,194128000.0,1240000.0,76055000.0,-49731000.0,-63089000.0,-0.17,-0.17,-71918000.0,-80964000.0
4,2025-12-31,27207000.0,-74898000.0,272459000.0,68217000.0,123863000.0,-67261000.0,-69380000.0,-0.20,-0.20,-43127000.0,-66747000.0


In [13]:
# PBR（Price-to-Book Ratio）やROE（Return on Equity）などの投資指標を計算
financial.calc_stock_investment_indicators(code=code, market=market)

取得件数: 457


,date,EV,reason,BPS,PBR,ROE,operating_income,basic_eps,diluted_eps
0,2021-12-31,NaN,no_price,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-12-31,NaN,no_price,NaN,NaN,-0.274244,-19794000.0,-0.08,-0.08
2,2023-12-31,NaN,no_price,NaN,NaN,-0.409456,-30842000.0,-0.12,-0.12
3,2024-12-31,NaN,NaN,0.364731,4.496462,-0.400523,-63089000.0,-0.17,-0.17
4,2025-12-31,NaN,NaN,0.204808,9.179350,-0.967025,-69380000.0,-0.20,-0.20


#### 技術報告書

In [15]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://d1io3yog0oux5.cloudfront.net/_1a8135fbabcb989da9f2c8410f0c92d9/urenergy/db/697/5645/file/20251231+LC+Property+Technical+Report+%28vF%29%28ws%29.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/20251231+LC+Property+Technical+Report+%28vF%29%28ws%29.pdf.md


'/workspace/data/20251231+LC+Property+Technical+Report+%28vF%29%28ws%29.pdf.md'

In [16]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://www.ur-energy.com/technical-reports#:~:text=%E3%82%B7%E3%83%A3%E3%83%BC%E3%83%AA%E3%83%BC%E7%9B%86%E5%9C%B0ISR%E3%82%A6%E3%83%A9%E3%83%B3%E3%83%97%E3%83%AD%E3%82%B8%E3%82%A7%E3%82%AF%E3%83%88%E3%80%81%E3%82%AB%E3%83%BC%E3%83%9C%E3%83%B3%E9%83%A1%E3%80%81%E3%83%AF%E3%82%A4%E3%82%AA%E3%83%9F%E3%83%B3%E3%82%B0%E5%B7%9E%E3%80%81%E3%82%A2%E3%83%A1%E3%83%AA%E3%82%AB%E5%90%88%E8%A1%86%E5%9B%BD%EF%BC%882024%E5%B9%B43%E6%9C%8811%E6%97%A5%EF%BC%89%E3%80%81%E4%BF%AE%E6%AD%A3%E6%B8%88%E3%81%BF%E3%80%81SK%201300%E5%90%91%E3%81%91",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/technical-reports.md


'/workspace/data/technical-reports.md'